In [ ]:
import shutil
import os

source_dir = '/kaggle/input/brain-tumor-mri-dataset/Training'
dest_dir = '/kaggle/working/brain-tumor-mri-dataset/Training'

shutil.copytree(source_dir, dest_dir)

In [ ]:
os.listdir(dest_dir)
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
import hashlib

def dhash(image, hash_size=8):
    resized = cv2.resize(image, (hash_size + 1, hash_size))
    diff = resized[:, 1:] > resized[:, :-1]
    return sum([2 ** i for (i, v) in enumerate(diff.flatten()) if v])

def remove_duplicates(data_dir):
    image_hashes = {}
    duplicates = []

    for category in os.listdir(data_dir):
        category_path = os.path.join(dest_dir, category)

        for img_name in os.listdir(category_path):
            img_path = os.path.join(category_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            img_hash = dhash(img)

            if img_hash in image_hashes:
                print(f"Duplicate found: {img_name} is a duplicate of {image_hashes[img_hash]}")
                duplicates.append(img_path)
            else:
                image_hashes[img_hash] = img_name

    for duplicate in duplicates:
        os.remove(duplicate)
        print(f"Deleted duplicate image: {duplicate}")

remove_duplicates(dest_dir)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

def load_and_preprocess_images(data_dir, target_size=(256, 256), categories=['notumor', 'glioma', 'meningioma', 'pituitary']):
    images = []
    labels = []
    valid_extensions = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
    
    if not os.path.exists(data_dir):
        raise FileNotFoundError(f"The directory {data_dir} does not exist.")
    
    # Loop through each category and load the images
    for idx, category in enumerate(categories):
        category_path = os.path.join(data_dir, category)
        if not os.path.exists(category_path):
            raise FileNotFoundError(f"The category directory {category_path} does not exist.")
        
        print(f"Processing category: {category}")
        for img_name in os.listdir(category_path):
            if any(img_name.lower().endswith(ext) for ext in valid_extensions):
                img_path = os.path.join(category_path, img_name)
                try:
                    img = load_img(img_path, color_mode='grayscale', target_size=target_size)
                    img_array = img_to_array(img)
                    images.append(img_array)
                    labels.append(idx)  # Label is the index of the category
                except Exception as e:
                    print(f"Error loading image {img_path}: {str(e)}")
    
    if not images:
        raise ValueError(f"No valid images found in the dataset.")
    
    print(f"Total images loaded: {len(images)}")
    
    
    images = np.array(images)
    labels = np.array(labels)
    
    return images, labels

def normalize_images(images):
    return (images / 127.5) - 1

def create_datasets(images, labels, batch_size=32, validation_split=0.2):
    if len(images) == 0 or len(labels) == 0:
        raise ValueError("The images or labels array is empty. Please check the data loading process.")
    
    
    labels = to_categorical(labels, num_classes=4)
    
    
    X_train, X_val, y_train, y_val = train_test_split(
        images, labels, test_size=validation_split, random_state=42, stratify=labels
    )
    
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(batch_size)
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)
    
    print(f"Training set size: {len(X_train)}")
    print(f"Validation set size: {len(X_val)}")
    
    return train_dataset, val_dataset


data_dir = '/kaggle/working/brain-tumor-mri-dataset/Training'
categories = ['notumor', 'glioma', 'meningioma', 'pituitary']


if __name__ == "__main__":
    images, labels = load_and_preprocess_images(data_dir, categories=categories)
    normalized_images = normalize_images(images)
    
    # Create train and validation datasets
    train_dataset, val_dataset = create_datasets(normalized_images, labels)
    print("Datasets created successfully.")


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_cnn_model(input_shape, num_classes):
    model = models.Sequential()
    
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model


cnn_model = build_cnn_model(input_shape=(256, 256, 1), num_classes=4)
cnn_model.summary()

history_cnn = cnn_model.fit(train_dataset, validation_data=val_dataset, epochs=10)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_classification_predictions(model, dataset, class_names):
    
    images, labels = next(iter(dataset))
    
    
    predictions = model.predict(images)
    predicted_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(labels, axis=1)

    
    plt.figure(figsize=(15, 15))
    for i in range(10):
        plt.subplot(5, 5, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap='gray')
        plt.title(f"True: {class_names[true_classes[i]]}\nPred: {class_names[predicted_classes[i]]}")
        plt.axis('off')
    plt.show()


class_names = ['notumor', 'glioma', 'meningioma', 'pituitary']
plot_classification_predictions(cnn_model, val_dataset, class_names)


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def evaluate_model(model, dataset, class_names):
    """
    Evaluate the model using various metrics and visualizations.
    
    Args:
        model: Trained keras model
        dataset: tf.data.Dataset containing validation/test data
        class_names: List of class names
    """
    
    y_pred = []
    y_true = []
    
    for images, labels in dataset:
        predictions = model.predict(images, verbose=0)
        y_pred.extend(np.argmax(predictions, axis=1))
        y_true.extend(np.argmax(labels, axis=1))
    
    # Convert to numpy arrays
    y_pred = np.array(y_pred)
    y_true = np.array(y_true)
    
    # Print classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

def plot_training_history(history):
    """
    Plot training history including accuracy and loss curves.
    
    Args:
        history: History object returned by model.fit()
    """
    # Plot training & validation accuracy
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training')
    plt.plot(history.history['val_accuracy'], label='Validation')
    plt.title('Model Accuracy over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training')
    plt.plot(history.history['val_loss'], label='Validation')
    plt.title('Model Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

def plot_sample_predictions(model, dataset, class_names, num_samples=10):
    """
    Plot sample predictions with confidence scores.
    
    Args:
        model: Trained keras model
        dataset: tf.data.Dataset containing validation/test data
        class_names: List of class names
        num_samples: Number of samples to display
    """
    images, labels = next(iter(dataset))
    predictions = model.predict(images, verbose=0)
    
    plt.figure(figsize=(15, num_samples * 2))
    for i in range(min(num_samples, len(images))):
        plt.subplot(num_samples, 1, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap='gray')
        
        true_class = class_names[np.argmax(labels[i])]
        pred_class = class_names[np.argmax(predictions[i])]
        confidence = np.max(predictions[i]) * 100
        
        plt.title(f'True: {true_class}\nPredicted: {pred_class} ({confidence:.1f}% confidence)')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()


class_names = ['notumor', 'glioma', 'meningioma', 'pituitary']
plot_training_history(history_cnn)
evaluate_model(cnn_model, val_dataset, class_names)
plot_sample_predictions(cnn_model, val_dataset, class_names)